[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Teoria_General_%28Jupyter.gen%29/Teoria_General_%28%28Jupyter.gen.gen%29/AST-13_Fisica_Estelar_Simulacion.ipynb)


# Simulación Computacional: Fisica Estelar
**Área:** Astrofisica y Cosmologia

Este cuaderno interactivo de Jupyter ejecuta numéricamente los modelos físicos descritos en el repositorio. Puedes modificar los parámetros iniciales y ejecutar las celdas para visualizar gráficos o animaciones interactivos.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def lane_emden(xi, y, n):
    """
    Ecuación de Lane-Emden para politropas: 
    (1/xi^2) * d/dxi(xi^2 * dtheta/dxi) = -theta^n
    Sistema de ODEs de primer orden:
    dtheta/dxi = u
    du/dxi = -theta^n - 2*u/xi
    """
    theta, u = y
    
    # Manejar singularidad en xi = 0
    if xi == 0:
        dtheta_dxi = 0
        du_dxi = -1/3
    else:
        # Prevenir potencias fraccionarias de negativos
        if theta < 0: theta = 0 
        dtheta_dxi = u
        du_dxi = -(theta**n) - (2/xi)*u
        
    return [dtheta_dxi, du_dxi]

# Condición límite en el centro estelar (xi = 0)
# theta(0) = 1, theta'(0) = 0
y0 = [1.0, 0.0]

# Pequeño desplazamiento inicial para evitar dividir por 0
xi_span = (1e-5, 10.0)
xi_eval = np.linspace(1e-5, 10.0, 500)

plt.figure(figsize=(9, 6))

# Índices politrópicos comunes
indices = [1.5, 3.0] # 1.5: convectiva (gas ideal), 3.0: radiativa (estrella típica Eddington)

for n in indices:
    sol = solve_ivp(lane_emden, xi_span, y0, args=(n,), t_eval=xi_eval, method='RK45')
    
    # Encontrar la superficie de la estrella (donde theta cae a 0)
    theta = sol.y[0]
    surface_idx = np.where(theta <= 0)[0]
    
    if len(surface_idx) > 0:
        idx = surface_idx[0]
        xi_star = sol.t[:idx]
        theta_star = theta[:idx]
    else:
        xi_star = sol.t
        theta_star = theta
        
    # Densidad proporcional a theta^n
    rho_norm = theta_star**n
    
    plt.plot(xi_star, rho_norm, label=f'Polítropa n={n}', lw=2)

plt.title('Perfil de Densidad Interna Estelar (Soluciones de Lane-Emden)')
plt.xlabel('Radio Normalizado $\\xi$')
plt.ylabel('Densidad Normalizada $\\rho / \\rho_c$')
plt.legend()
plt.grid(True)
plt.show()